# Запуск расчёта метрик MT-модели (osiris): модель с канальными башнями и модель с канальной поправкой

Один лаунчер для обеих версий: набор выходов определяется по `<mdl_name>_mt_config.json`.
- Модель с **канальными башнями** (предыдущая версия): выходы `cr` (башня канала строки), при наличии общей головы `cr_general`, и `ctr`.
- Модель с **канальной поправкой**: выходы `cr_general`, `cr_adj`, `ctr`.

Для сравнения старой и новой модели на одних и тех же строках включите `SAVE_PREDS = True` в обоих прогонах (одна таблица, одинаковые `BATCH_SIZE`, `NUM_WORKERS`, число процессов, `FILE_FRACTION`, `SEED`) и вызовите `compare_runs(...)` в последней ячейке.

Порядок: 1) импорты → 2) Kerberos → 3) параметры → 4) preflight → 5) задача → 6) запуск → 7) мониторинг и результаты → 8) сравнение двух прогонов.

In [5]:
import os
import sys
import re
import glob
import json
import time
import shutil
import getpass
import subprocess

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/opt/clients"))
import osiris

## 2. Kerberos / Hadoop (один раз за сессию)

In [ ]:
NFS = "/home/datalab/nfs"
PREPARE_ENV = True   # False, если ticket и конфиги уже подготовлены

if PREPARE_ENV:
    for name in ("hadoop", "nss", "krb5cc", "krb5.conf"):
        path = f"{NFS}/{name}"
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    shutil.copytree("/etc/hadoop/conf", f"{NFS}/hadoop", dirs_exist_ok=True)
    shutil.copy("/etc/krb5.conf", f"{NFS}/krb5.conf")
    subprocess.run(["kinit"], input=getpass.getpass("Пароль для kinit: ") + "\n", text=True, check=True)
    ticket = max(glob.glob("/tmp/krb5cc_*"), key=os.path.getmtime)
    shutil.copy(ticket, f"{NFS}/krb5cc")
    os.makedirs(f"{NFS}/nss", exist_ok=True)
    shutil.copy("/etc/passwd", f"{NFS}/nss/")
    print("Kerberos/Hadoop подготовлены, ticket:", ticket)

## 3. Параметры

In [ ]:
# ============ РЕЖИМ ============
RUN_MODE = "full"        # "check" — малая доля файлов; "full" — вся выборка
DRY_RUN = False            # True: только печать аргументов, задача не создаётся

PRESETS = {
    "check": dict(file_fraction=0.0002, max_batches=0, num_nodes=2, num_gpus=4, suffix="_check"),
    "full":  dict(file_fraction=1.0,  max_batches=0, num_nodes=3, num_gpus=4, suffix=""),
}

# ============ ПУТИ ============
METRICS_SCRIPT_PATH = "/home/datalab/nfs/Omni/SCORE OMNI/MDMH score/CR CHNLS CTR EXPERTS score/mt_ddp_metrics.py"   # положите скрипт рядом с обучающим или укажите путь
LOCAL_ROOT = "/home/datalab/nfs/Omni/"
CONFIG_PATH = "/home/datalab/nfs/Omni/configs_weights/omni_config.json"
DATASET_PATH = ""
WEIGHTS_DIR = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/weights"
METRICS_DIR = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/metrics"

# ============ МОДЕЛЬ ДЛЯ ОЦЕНКИ ============
# Версия с канальными башнями: например MDL_NAME = "omni_val_cr_ctr_mt_smoke" (имя модели из того обучения)
# Версия с поправкой:           например MDL_NAME = "omni_val_cr_ctr_mt_adapter_check"
MDL_NAME = "omni_ML_DCN_val_cr_ctr_mt_star_concat_small_head"

LAST_EPOCH = 1                                           # индекс последней эпохи (EPOCHS - 1)
CKPT_NAME = f"{MDL_NAME}_ep{LAST_EPOCH}_weights.pt"      # лучше *_weights.pt
MT_CONFIG_NAME = f"{MDL_NAME}_mt_config.json"            # сохраняется скриптом обучения рядом с весами

# ============ ДАННЫЕ ДЛЯ ОЦЕНКИ ============
TRAIN_TABLE = "sdi_cr_ctr_dataset_val_fin"               # только для предупреждения об оценке на обучающей выборке
VAL_TABLE = ""                                           # ОБЯЗАТЕЛЬНО: таблица для оценки
TST_TABLE = ""
OOT_TABLE = "sdi_cr_ctr_dataset_oot"
EVAL_SPLITS = "oot"                                      # val,tst,oot через запятую
DATASET_SIZE = 202_666_520                               # число строк сплита (только для прогресс-бара)
DATA_FORMAT = "raw"
TARGET_COLUMNS = "trgt_fin,trgt_ctr"                     # в том же порядке, что при обучении

# ============ МЕТРИКИ ============
HEADS = "auto"                                           # auto или список: cr,cr_general,cr_adj,ctr
MODES = "general,product,channel,product_channel"
N_BOOT = 200
BOOT_MAX_ROWS = 300000
NO_COMPARE = True
SAVE_PREDS = False                                       # True: сохранить предсказания для compare_runs (старая vs новая модель)
RESULT_NAME = "metrics_ml_dcn_star_concat_small"

# ============ ПРОЧЕЕ ============
BATCH_SIZE = 128
SEED = 42
NUM_WORKERS = 2
USE_EMB_FLAGS = """{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }"""
EVK_WEIGHTS_PATH = "evk_weights.pt"
DLT_WEIGHTS_PATH = "dlt_weights.pt"
POOL = "b2c"

# ============ ПРОИЗВОДНЫЕ ============
assert RUN_MODE in PRESETS, f"RUN_MODE должен быть одним из {list(PRESETS)}"
P = PRESETS[RUN_MODE]
FILE_FRACTION, MAX_BATCHES = P["file_fraction"], P["max_batches"]
NUM_NODES, NUM_GPUS = P["num_nodes"], P["num_gpus"]
NUM_PROCESS = NUM_NODES * NUM_GPUS
RESULT_TAG = RESULT_NAME + P["suffix"]
print(f"mode={RUN_MODE} model={MDL_NAME} ckpt={CKPT_NAME} splits={EVAL_SPLITS} procs={NUM_PROCESS} file_fraction={FILE_FRACTION}")

## 4. Preflight

Проверяет скрипт метрик, чекпоинт и `mt_config.json`, определяет выходы модели, пары для сравнения внутри модели, число колонок таргета и таблицы для сплитов.

In [ ]:
def _fail(msg):
    raise AssertionError("PREFLIGHT: " + msg)

def preflight():
    try:
        json.loads(USE_EMB_FLAGS)
    except Exception as e:
        _fail(f"USE_EMB_FLAGS не парсится как JSON: {e}")
    if not os.path.exists(METRICS_SCRIPT_PATH):
        _fail(f"нет METRICS_SCRIPT_PATH={METRICS_SCRIPT_PATH}")
    text = open(METRICS_SCRIPT_PATH, encoding="utf-8").read()
    for tok in ("--mt-config", "--heads", "collect_predictions", "compare_table", "build_pairs", "--save-preds"):
        if tok not in text:
            _fail(f"в {METRICS_SCRIPT_PATH} нет '{tok}': это не актуальный mt_ddp_metrics.py")
    ckpt = f"{WEIGHTS_DIR}/{CKPT_NAME}"
    if not os.path.exists(ckpt):
        cands = sorted(glob.glob(f"{WEIGHTS_DIR}/{MDL_NAME}_ep*_weights.pt"))
        _fail(f"нет чекпоинта {ckpt}. Найдено для модели: {[os.path.basename(c) for c in cands]}")
    mt_path = f"{WEIGHTS_DIR}/{MT_CONFIG_NAME}"
    if not os.path.exists(mt_path):
        _fail(f"нет {mt_path}: файл сохраняет скрипт обучения рядом с весами")
    cfg = json.load(open(mt_path))
    cps = cfg["channels_per_task"]
    no_towers = cfg.get("tower_type", "per_channel") == "none"
    heads = {t: t for t in cfg["tasks"] if not (no_towers and cps[t] > 1)}
    heads.update({f"{t}_general": t for t in cfg.get("general_tasks", [])})
    heads.update({f"{t}_adj": t for t in cfg.get("adapter_tasks", [])})
    #heads.update({f"{t}_ws": t for t in cfg.get("ws_tasks", [])})
    if HEADS != "auto":
        sel = [h.strip() for h in HEADS.split(",") if h.strip()]
        unknown = [h for h in sel if h not in heads]
        if unknown:
            _fail(f"HEADS: неизвестные выходы {unknown}; доступны {list(heads)}")
        heads = {h: heads[h] for h in sel}
    cols = [c.strip() for c in TARGET_COLUMNS.split(",")]
    need = max(cfg["target_cols"][t] for t in heads.values()) + 1
    if len(cols) < need:
        _fail(f"TARGET_COLUMNS={cols} короче, чем нужно ({need} колонок) для выходов {list(heads)}")
    tables = {"val": VAL_TABLE, "tst": TST_TABLE, "oot": OOT_TABLE}
    for s in [x.strip() for x in EVAL_SPLITS.split(",") if x.strip()]:
        if s not in tables:
            _fail(f"неизвестный сплит {s}; доступны {list(tables)}")
        if not tables[s]:
            _fail(f"для сплита '{s}' не задана таблица ({s.upper()}_TABLE)")
        if tables[s] == TRAIN_TABLE:
            print(f"WARN: сплит '{s}' = обучающая таблица {TRAIN_TABLE}: метрики будут оптимистичны")
    pairs = [f"{t}_adj vs {t}_general" for t in cfg.get("adapter_tasks", []) if f"{t}_adj" in heads and f"{t}_general" in heads]
    pairs += [f"{t} vs {t}_general" for t in cfg.get("general_tasks", []) if t in heads and f"{t}_general" in heads]
    kind = "канальные башни" if any(h == "cr" for h in heads) else "общая голова + поправка"
    print("тип модели по конфигу:", kind)
    print("выходы для оценки:", heads)
    print("сравнение внутри модели:", pairs if (pairs and not NO_COMPARE) else "не будет (нет пары выходов или NO_COMPARE)")
    if SAVE_PREDS:
        print(f"предсказания будут сохранены: {METRICS_DIR}/preds_{MDL_NAME}_<сплит>.npz")
    print("PREFLIGHT: OK")
    return heads

HEADS_RESOLVED = preflight()

## 5. Задача osiris

In [ ]:
def build_script_args():
    a = [
        "--num_machines", f"{NUM_NODES}",
        "--num_processes", f"{NUM_PROCESS}",
        "--main_process_ip", "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port", "$(MASTER_PORT)",
        "--machine_rank", "$(RANK)",
        f"{METRICS_SCRIPT_PATH}",
        "--local-root", str(LOCAL_ROOT),
        "--config-path", str(CONFIG_PATH),
        "--mdl-name", str(MDL_NAME),
        "--chkpt-name", str(CKPT_NAME),
        "--mt-config", str(MT_CONFIG_NAME),
        "--target-columns", str(TARGET_COLUMNS),
        "--heads", str(HEADS),
        "--eval-splits", str(EVAL_SPLITS),
        "--data-format", str(DATA_FORMAT),
        "--dataset-size", str(DATASET_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--num-workers", str(NUM_WORKERS),
        "--file-fraction", str(FILE_FRACTION),
        "--modes", str(MODES),
        "--n-boot", str(N_BOOT),
        "--boot-max-rows", str(BOOT_MAX_ROWS),
        "--use-emb-flags", str(USE_EMB_FLAGS),
        "--evk-weights-path", EVK_WEIGHTS_PATH,
        "--dlt-weights-path", DLT_WEIGHTS_PATH,
        "--result-name", str(RESULT_TAG),
    ]
    if DATASET_PATH:
        a += ["--dataset-path", DATASET_PATH]
    for flag, val in (("--val-table", VAL_TABLE), ("--tst-table", TST_TABLE), ("--oot-table", OOT_TABLE)):
        if val:
            a += [flag, val]
    if MAX_BATCHES:
        a += ["--max-batches", str(MAX_BATCHES)]
    if NO_COMPARE:
        a += ["--no-compare"]
    if SAVE_PREDS:
        a += ["--save-preds"]
    return a


def create_job(job_name, num_gpus):
    return osiris.create(
        name=f"sc-{job_name}",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
        restart=True,
        pool=POOL,
        script=METRICS_SCRIPT_PATH,
        command=["accelerate", "launch"],
        args=build_script_args(),
        envs={
            "PYTHONPATH": "",
            "OMP_NUM_THREADS": "30",
            "NCCL_DEBUG": "INFO",
            "HADOOP_CONF_DIR": "/home/datalab/nfs/hadoop",
            "KRB5_CONFIG": "/home/datalab/nfs/krb5.conf",
            "KRB5CCNAME": "/home/datalab/nfs/krb5cc",
            "ARROW_LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "HADOOP_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LD_PRELOAD": "/usr/lib64/libnss_wrapper.so",
            "NSS_WRAPPER_PASSWD": "/home/datalab/nfs/nss/passwd",
            "NSS_WRAPPER_GROUP": "/etc/group",
            "CLASSPATH": f":/home/datalab/nfs/hadoop:" + os.environ["CLASSPATH"].split(":", 2)[-1],
            "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
            "NCCL_SOCKET_FAMILY": "AF_INET",
            "GLOO_SOCKET_FAMILY": "AF_INET",
            "NCCL_SOCKET_IFNAME": "eth0",
            "GLOO_SOCKET_IFNAME": "eth0",
            "NCCL_IB_DISABLE": "1",
            "NCCL_P2P_DISABLE": "1",
            "NCCL_SHM_DISABLE": "1",
        },
        num_nodes=NUM_NODES,
        num_gpus=num_gpus,
        type="pytorchjob",
    )

print(" ".join(f"'{x}'" if (" " in x or x[:1] in "{[") else x for x in build_script_args()))

## 6. Запуск

`DRY_RUN = True` ничего не создаёт. После `DRY_RUN = False` выполните эту ячейку и следующую.

In [ ]:
import time 
time.sleep(30 * 60)

In [ ]:
JOB_NAME = f"mt-metrics-{RUN_MODE}"

if DRY_RUN:
    job_name = None
    print("DRY_RUN: задача не создавалась")
else:
    HEADS_RESOLVED = preflight()
    job = create_job(JOB_NAME, NUM_GPUS)
    print(job)
    job_name = job["job"]

## 7. Мониторинг, разбор логов и результаты

In [ ]:
def _find_job(name):
    for j in osiris.list()["jobs"]:
        if j["job"] == name:
            return j
    return None


def check_report(text):
    issues = []
    for pat in (r"\[EVAL\] выходы:.*", r"\[CKPT\] Загружено.*", r"\[EVAL\] строк всего:.*", r"\[METRICS\] Saved:.*"):
        m = re.search(pat, text)
        print(m.group(0)[:300] if m else f"(нет строки по шаблону {pat})")
        if not m and "Saved" in pat:
            issues.append("нет строки [METRICS] Saved: результат не записан")
    for w in re.findall(r"\[WARN\][^\r\n]*|\[CKPT\] WARNING[^\r\n]*", text):
        issues.append(w[:250])
    if "Traceback" in text:
        issues.append(f"в логе есть Traceback ({text.count('Traceback')})")
    if "в чекпоинте нет весов MT-головы" in text:
        issues.append("в чекпоинте нет весов mt.* (не тот чекпоинт или конфиг)")
    return issues


def show_results(split=None, tag=None):
    split = split or EVAL_SPLITS.split(",")[0].strip()
    tag = tag or RESULT_TAG
    files = sorted(glob.glob(f"{METRICS_DIR}/mt_*_{split}_{tag}.xlsx"), key=os.path.getmtime)
    if not files:
        print("нет файлов результатов:", f"{METRICS_DIR}/mt_*_{split}_{tag}.xlsx")
        return None
    path = files[-1]
    print("файл:", path, time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path))))
    xl = pd.read_excel(path, sheet_name=None)
    m = xl["metrics"]
    show = [c for c in ("model", "slice_name", "communications", "avg_cr", "gini", "pr_auc", "logloss", "ece") if c in m.columns]
    print("\nметрики (общие и по каналам):")
    print(m[m["slice_type"].isin(["general", "channel"])][show].round(4).to_string(index=False))
    if "compare" in xl:
        c = xl["compare"]
        print("\nсравнение выходов внутри модели, a - b (95% бутстрэп-интервал):")
        print(c[["пара", "канал", "строк", "d_roc_auc", "d_roc_lo", "d_roc_hi", "вывод_roc", "d_pr_auc", "d_pr_lo", "d_pr_hi", "вывод_pr"]].round(4).to_string(index=False))
    return xl


if job_name:
    my_job = _find_job(job_name)
    while my_job is None or my_job["state"] not in ("Succeeded", "Failed"):
        my_job = _find_job(job_name)
        if my_job is not None:
            print(my_job["state"], end="\r")
        time.sleep(5)
    print("Состояние:", my_job["state"])
    try:
        out = osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=20000)
        text = out if isinstance(out, str) else (str(out) if out is not None else "")
    except Exception as e:
        text = ""
        print("не удалось получить логи автоматически:", e)
    if text:
        issues = check_report(text)
        print("\nЗамечания:" if issues else "\nЗамечаний нет")
        for i in issues:
            print(" -", i)
    if my_job["state"] == "Succeeded":
        xl = show_results()
else:
    print("задача не создавалась (DRY_RUN)")

In [ ]:
from pprint import pprint

In [ ]:
# логи мастера вручную (ищите [EVAL], [CKPT], [WARN], Traceback)
if job_name:
    pprint(osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=300))
else:
    print("задача не создавалась (DRY_RUN)")

## 8. Сравнение двух прогонов (например, модель с башнями и модель с поправкой)

Оба прогона должны быть запущены с `SAVE_PREDS = True` на **одной и той же таблице** и с одинаковыми `BATCH_SIZE`, `NUM_WORKERS`, числом процессов, `FILE_FRACTION` и `SEED`. Функция проверяет, что таргеты и каналы в двух файлах совпадают построчно; если нет, сравнение не выполняется. Разница считается как `B − A` (новая минус старая) с парным бутстрэпом.

In [ ]:
def _load_helpers():
    text = open(METRICS_SCRIPT_PATH, encoding="utf-8").read()
    block = re.search(r"# <<<HELPERS\n(.*?)# >>>HELPERS", text, re.S).group(1)
    ns = {"np": np, "pd": pd}
    exec(block, ns)
    return ns


def compare_runs(mdl_a, head_a, mdl_b, head_b, split="val", target_idx=0, n_boot=200, max_rows=300000,
                 channel_names=None, label_a="A", label_b="B"):
    """Парное сравнение выхода head_b модели mdl_b с выходом head_a модели mdl_a. target_idx: 0 для CR, 1 для CTR."""
    channel_names = channel_names or {0: "mass", 1: "phygital", 2: "premier", 3: "sbol"}
    A = np.load(f"{METRICS_DIR}/preds_{mdl_a}_{split}.npz")
    B = np.load(f"{METRICS_DIR}/preds_{mdl_b}_{split}.npz")
    names_a, names_b = [str(x) for x in A["names"]], [str(x) for x in B["names"]]
    for h, nm, mdl in ((head_a, names_a, mdl_a), (head_b, names_b, mdl_b)):
        if h not in nm:
            raise ValueError(f"у модели {mdl} нет выхода {h}; есть {nm}")
    if A["T"].shape != B["T"].shape or not np.array_equal(A["T"], B["T"]) or not np.array_equal(A["G"], B["G"]):
        raise ValueError("строки двух прогонов не совпадают (разный порядок или состав): парное сравнение невозможно. "
                         "Прогоны должны использовать одну таблицу, BATCH_SIZE, NUM_WORKERS, число процессов, FILE_FRACTION и SEED")
    pa, pb = A["P"][:, names_a.index(head_a)], B["P"][:, names_b.index(head_b)]
    P = np.stack([pb, pa], axis=1)
    nm = [f"{label_b}:{head_b}", f"{label_a}:{head_a}"]
    ns = _load_helpers()
    table = ns["compare_table"](P, A["T"], A["G"], nm, {nm[0]: "t", nm[1]: "t"}, {"t": target_idx},
                                [(nm[0], nm[1])], lambda c: channel_names.get(int(c), str(int(c))), n_boot=n_boot, max_rows=max_rows)
    cols = ["пара", "канал", "строк", "roc_auc_a", "roc_auc_b", "d_roc_auc", "d_roc_lo", "d_roc_hi", "вывод_roc",
            "d_pr_auc", "d_pr_lo", "d_pr_hi", "вывод_pr"]
    print(table[cols].round(4).to_string(index=False))
    return table

# пример: старая модель с башнями (A) против новой с поправкой (B), CR:
# t = compare_runs("omni_val_cr_ctr_mt_smoke", "cr", "omni_val_cr_ctr_mt_adapter_check", "cr_adj", split="val")

In [20]:
# удаление задачи
if job_name: osiris.delete(job_name)